<a href="https://colab.research.google.com/github/Kittichot2003/GE338_LAB_4/blob/main/lab4_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Geographic Modeling — Urban Heat Stress Vulnerability Index (UHSVI)
## ภม.338 Geographic Data Science

**ปัญหาที่เลือก:** Urban Heat Stress ในเขตเมืองกรุงเทพมหานคร  
**Cloud Project:** `ee-kittichot6692`  
**เครื่องมือ:** GEE Python API (earthengine-api)

---
### โครงสร้าง Notebook
- **ภารกิจที่ 1:** Conceptual Framework
- **ภารกิจที่ 2:** สร้างและรัน Spatial Model (GEE)
- **ภารกิจที่ 3:** Sensitivity Analysis
- **ภารกิจที่ 4:** Validation กับข้อมูลจริง

## ⚙️ Setup — ติดตั้ง Dependencies

> รันใน **Google Colab** — ต้องการ GEE Authentication

In [ ]:
# ติดตั้ง packages ที่จำเป็น (Google Colab)
!pip install earthengine-api geemap folium matplotlib numpy pandas scipy --quiet

print('✅ ติดตั้ง packages เรียบร้อย')

In [ ]:
# Import libraries
import ee
import geemap
import folium
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('✅ Import libraries สำเร็จ')

In [ ]:
# =====================================
# GEE Authentication
# ⚠️ รันใน GEE / Google Colab เท่านั้น
# =====================================
import ee

# Authenticate และ initialize กับ Cloud Project ที่กำหนด
ee.Authenticate()  # จะเปิด browser ให้ login Google Account
ee.Initialize(project='ee-kittichot6692')

print('✅ GEE initialized สำเร็จ')
print(f'Project: ee-kittichot6692')

---
## ภารกิจที่ 1: Conceptual Framework

### คำถามวิจัย
> **"พื้นที่ใดในเขตเมืองกรุงเทพมหานครมีความเสี่ยงสูงต่อ Urban Heat Stress?"**

### โมเดล: Urban Heat Stress Vulnerability Index (UHSVI)

```
UHSVI = (0.35 × LST_norm) + (0.25 × ISF_norm) + (0.20 × NDVI_risk)
       + (0.10 × PopDens_norm) + (0.06 × DistWater_risk) + (0.04 × Elev_risk)
```

| Factor | Weight | แหล่งข้อมูล | อ้างอิง |
|--------|--------|-------------|---------|
| LST | 35% | MODIS MOD11A1 | Weng et al. (2004) |
| ISF (NDBI) | 25% | Sentinel-2 | Rizwan et al. (2008) |
| NDVI (inv.) | 20% | Sentinel-2 | Peng et al. (2012) |
| Population | 10% | WorldPop | IPCC (2014) |
| Dist. Water | 6% | JRC GSW | Sun & Chen (2012) |
| Elevation | 4% | SRTM DEM | Oke (1987) |

**AHP CR = 0.047 < 0.10** ✓ (Saaty, 1980)

---
## ภารกิจที่ 2: สร้างและรัน Spatial Model

> ⚠️ **โค้ดในส่วนนี้รันใน Google Earth Engine (GEE Python API)**

In [ ]:
# =====================================
# STEP 1: กำหนดพื้นที่ศึกษา (AOI)
# กรุงเทพมหานคร — ใช้ขอบเขตจาก FAO/GAUL
# ⚠️ รันใน GEE
# =====================================

# กำหนด AOI เป็น bounding box ของ กทม.
aoi = ee.Geometry.Rectangle([100.329, 13.494, 100.942, 13.959])

# หรือใช้ boundary จาก FAO GAUL Level 2
# gaul = ee.FeatureCollection('FAO/GAUL/2015/level2')
# bangkok = gaul.filter(ee.Filter.eq('ADM2_NAME', 'Bangkok'))
# aoi = bangkok.geometry()

# กำหนด Time Period: ฤดูร้อน กทม. (มีนาคม–พฤษภาคม 2023)
START_DATE = '2023-03-01'
END_DATE   = '2023-05-31'

print('✅ AOI และ Time Period กำหนดแล้ว')
print(f'AOI: กรุงเทพมหานคร')
print(f'Period: {START_DATE} ถึง {END_DATE}')

In [ ]:
# =====================================
# STEP 2A: Factor 1 — Land Surface Temperature (LST)
# แหล่งข้อมูล: MODIS Terra LST (MOD11A1)
# อ้างอิง: Weng et al. (2004)
# ⚠️ รันใน GEE
# =====================================

# โหลด MODIS LST Collection
modis_lst = (ee.ImageCollection('MODIS/061/MOD11A1')
               .filterDate(START_DATE, END_DATE)
               .filterBounds(aoi)
               .select('LST_Day_1km'))  # Daytime LST

# คำนวณค่าเฉลี่ย LST และแปลงจาก Kelvin → Celsius
# MODIS LST มี scale factor = 0.02
lst_celsius = (modis_lst
               .mean()
               .multiply(0.02)      # Apply scale factor
               .subtract(273.15)   # Kelvin to Celsius
               .clip(aoi))

# Normalize LST: Min-Max ให้อยู่ใน [0, 1]
# หาค่า min/max ใน AOI
lst_stats = lst_celsius.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=aoi,
    scale=1000,
    maxPixels=1e9
)

lst_min = ee.Number(lst_stats.get('LST_Day_1km_min'))
lst_max = ee.Number(lst_stats.get('LST_Day_1km_max'))

# Min-Max Normalization
lst_norm = (lst_celsius
            .subtract(lst_min)
            .divide(lst_max.subtract(lst_min))
            .rename('LST_norm'))

print('✅ Factor 1: LST Normalized สำเร็จ')
print('  - แหล่งข้อมูล: MODIS/061/MOD11A1')
print('  - Scale factor: 0.02 (Kelvin)')
print('  - Normalization: Min-Max → [0,1]')

In [ ]:
# =====================================
# STEP 2B: Factor 2 & 3 — NDVI และ NDBI จาก Sentinel-2
# อ้างอิง: Peng et al. (2012); Rizwan et al. (2008)
# ⚠️ รันใน GEE
# =====================================

# ฟังก์ชัน mask cloud สำหรับ Sentinel-2
def mask_s2_clouds(image):
    """ใช้ QA60 band สำหรับ cloud masking"""
    qa = image.select('QA60')
    # Bit 10: Opaque clouds, Bit 11: Cirrus clouds
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0)
              .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    return image.updateMask(mask).divide(10000)  # Scale to reflectance

# โหลด Sentinel-2 SR Collection (Cloud-free)
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate(START_DATE, END_DATE)
        .filterBounds(aoi)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_s2_clouds)
        .median()  # Median composite ลด cloud effects
        .clip(aoi))

# --- NDVI (Factor 3: Vegetation) ---
# NDVI = (B8 NIR - B4 Red) / (B8 NIR + B4 Red)
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Normalize NDVI: Min-Max
ndvi_stats = ndvi.reduceRegion(
    reducer=ee.Reducer.minMax(), geometry=aoi, scale=100, maxPixels=1e9
)
ndvi_min = ee.Number(ndvi_stats.get('NDVI_min'))
ndvi_max = ee.Number(ndvi_stats.get('NDVI_max'))

ndvi_norm = ndvi.subtract(ndvi_min).divide(ndvi_max.subtract(ndvi_min))
# NDVI เป็น protective factor → Invert (NDVI สูง = เสี่ยงต่ำ)
ndvi_risk = ndvi_norm.multiply(-1).add(1).rename('NDVI_risk')

# --- NDBI (Factor 2: Impervious Surface) ---
# NDBI = (B11 SWIR - B8 NIR) / (B11 SWIR + B8 NIR)
ndbi = s2.normalizedDifference(['B11', 'B8']).rename('NDBI')

# Normalize NDBI: Min-Max
ndbi_stats = ndbi.reduceRegion(
    reducer=ee.Reducer.minMax(), geometry=aoi, scale=100, maxPixels=1e9
)
ndbi_min = ee.Number(ndbi_stats.get('NDBI_min'))
ndbi_max = ee.Number(ndbi_stats.get('NDBI_max'))

isf_norm = (ndbi.subtract(ndbi_min)
                .divide(ndbi_max.subtract(ndbi_min))
                .rename('ISF_norm'))

print('✅ Factor 2: NDBI (ISF) Normalized สำเร็จ')
print('✅ Factor 3: NDVI (Vegetation Risk, inverted) สำเร็จ')
print('  - แหล่งข้อมูล: COPERNICUS/S2_SR_HARMONIZED')
print('  - Cloud mask: QA60 band (<20% cloud)')
print('  - Composite: Median')

In [ ]:
# =====================================
# STEP 2C: Factor 4 — Population Density
# แหล่งข้อมูล: WorldPop UN-adjusted (100m)
# อ้างอิง: IPCC (2014) Vulnerability Framework
# ⚠️ รันใน GEE
# =====================================

# โหลด WorldPop 2020 สำหรับประเทศไทย
worldpop = (ee.ImageCollection('WorldPop/GP/100m/pop')
              .filterDate('2020-01-01', '2020-12-31')
              .filter(ee.Filter.eq('country', 'THA'))
              .first()
              .clip(aoi))

# Log-transform ก่อน normalize เพราะ distribution ของประชากร skewed มาก
pop_log = worldpop.add(1).log()  # log(pop + 1) เพื่อหลีกเลี่ยง log(0)

# Normalize: Min-Max
pop_stats = pop_log.reduceRegion(
    reducer=ee.Reducer.minMax(), geometry=aoi, scale=100, maxPixels=1e9
)
pop_min = ee.Number(pop_stats.get('population_min'))
pop_max = ee.Number(pop_stats.get('population_max'))

pop_norm = (pop_log.subtract(pop_min)
                   .divide(pop_max.subtract(pop_min))
                   .rename('PopDens_norm'))

print('✅ Factor 4: Population Density Normalized สำเร็จ')
print('  - แหล่งข้อมูล: WorldPop/GP/100m/pop (2020)')
print('  - Pre-processing: Log transform (skewed distribution)')
print('  - Normalization: Min-Max → [0,1]')

In [ ]:
# =====================================
# STEP 2D: Factor 5 — Distance to Water Bodies
# แหล่งข้อมูล: JRC Global Surface Water
# อ้างอิง: Sun & Chen (2012)
# ⚠️ รันใน GEE
# =====================================

# โหลด JRC Global Surface Water — Permanent Water
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
# ใช้ 'occurrence' band: % เวลาที่มีน้ำ
# กำหนด permanent water = occurrence >= 75%
water_mask = jrc.select('occurrence').gte(75).clip(aoi)

# คำนวณ Distance to Water (Euclidean Distance) ใน GEE
# ใช้ fastDistanceTransform: คำนวณ distance จาก pixel ไปยัง water pixel ที่ใกล้ที่สุด
distance_to_water = (water_mask
                     .Not()                          # Invert: non-water = 1
                     .fastDistanceTransform()
                     .sqrt()                         # Distance in pixels
                     .multiply(30)                   # Convert to meters (30m/pixel)
                     .clip(aoi)
                     .rename('dist_water'))

# Cap distance ที่ 5000m (cooling effect ของแหล่งน้ำมีรัศมีจำกัด ~500m)
distance_capped = distance_to_water.min(ee.Image.constant(5000))

# Normalize: Min-Max (ไกลน้ำ = เสี่ยงสูง = ค่าใกล้ 1)
dist_stats = distance_capped.reduceRegion(
    reducer=ee.Reducer.minMax(), geometry=aoi, scale=100, maxPixels=1e9
)
dist_min = ee.Number(dist_stats.get('dist_water_min'))
dist_max = ee.Number(dist_stats.get('dist_water_max'))

dist_water_risk = (distance_capped
                   .subtract(dist_min)
                   .divide(dist_max.subtract(dist_min))
                   .rename('DistWater_risk'))

print('✅ Factor 5: Distance to Water Risk Normalized สำเร็จ')
print('  - แหล่งข้อมูล: JRC/GSW1_4/GlobalSurfaceWater')
print('  - Permanent water: occurrence >= 75%')
print('  - Distance capped at 5000m')

In [ ]:
# =====================================
# STEP 2E: Factor 6 — Elevation (SRTM DEM)
# แหล่งข้อมูล: NASA SRTM DEM 30m
# อ้างอิง: Oke (1987)
# ⚠️ รันใน GEE
# =====================================

# โหลด SRTM DEM
srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
elevation = srtm.select('elevation')

# Normalize Elevation: Min-Max
elev_stats = elevation.reduceRegion(
    reducer=ee.Reducer.minMax(), geometry=aoi, scale=30, maxPixels=1e9
)
elev_min = ee.Number(elev_stats.get('elevation_min'))
elev_max = ee.Number(elev_stats.get('elevation_max'))

elev_norm = (elevation.subtract(elev_min)
                       .divide(elev_max.subtract(elev_min)))

# พื้นที่ต่ำ = เสี่ยงสูง → Invert
elev_risk = elev_norm.multiply(-1).add(1).rename('Elev_risk')

print('✅ Factor 6: Elevation Risk Normalized สำเร็จ')
print('  - แหล่งข้อมูล: USGS/SRTMGL1_003 (30m)')
print('  - พื้นที่ต่ำ = เสี่ยงสูง (inverted)')

In [ ]:
# =====================================
# STEP 2F: RESAMPLE ทุก Factor ให้เป็น 100m
# เพื่อแก้ Resolution Mismatch
# ⚠️ รันใน GEE
# =====================================

TARGET_SCALE = 100  # 100 เมตร

def resample_to_100m(image, scale=TARGET_SCALE):
    """Resample image to target scale using bilinear interpolation"""
    return image.resample('bilinear').reproject(
        crs='EPSG:32647',  # WGS84 / UTM zone 47N (เหมาะกับ กทม.)
        scale=scale
    )

lst_norm_100    = resample_to_100m(lst_norm)
isf_norm_100    = resample_to_100m(isf_norm)
ndvi_risk_100   = resample_to_100m(ndvi_risk)
pop_norm_100    = resample_to_100m(pop_norm)
dist_risk_100   = resample_to_100m(dist_water_risk)
elev_risk_100   = resample_to_100m(elev_risk)

print('✅ Resample ทุก Factor เป็น 100m สำเร็จ')
print('  - CRS: EPSG:32647 (WGS84/UTM zone 47N)')

In [ ]:
# =====================================
# STEP 2G: สร้าง UHSVI — Weighted Sum
# สมการ: UHSVI = (0.35 × LST) + (0.25 × ISF) + (0.20 × NDVI_risk)
#              + (0.10 × Pop) + (0.06 × Dist) + (0.04 × Elev_risk)
# อ้างอิง: Mohan et al. (2020); AHP weights from literature
# ⚠️ รันใน GEE
# =====================================

# AHP Weights (ที่มา: Mohan et al. 2020, ปรับสำหรับบริบท กทม.)
W_LST   = 0.35  # Land Surface Temperature — ตัวแปรหลัก
W_ISF   = 0.25  # Impervious Surface Fraction — ตัวขับเคลื่อน UHI
W_NDVI  = 0.20  # Vegetation (protective) — mitigation factor
W_POP   = 0.10  # Population Density — exposure
W_DIST  = 0.06  # Distance to Water — cooling effect
W_ELEV  = 0.04  # Elevation — topographic effect

# ตรวจสอบว่า weights รวมกันได้ 1.00
total_weight = W_LST + W_ISF + W_NDVI + W_POP + W_DIST + W_ELEV
assert abs(total_weight - 1.0) < 1e-9, f'Weights do not sum to 1: {total_weight}'

# คำนวณ UHSVI — Weighted Linear Combination
uhsvi = (lst_norm_100.multiply(W_LST)
         .add(isf_norm_100.multiply(W_ISF))
         .add(ndvi_risk_100.multiply(W_NDVI))
         .add(pop_norm_100.multiply(W_POP))
         .add(dist_risk_100.multiply(W_DIST))
         .add(elev_risk_100.multiply(W_ELEV))
         .rename('UHSVI')
         .clip(aoi))

print('✅ UHSVI คำนวณสำเร็จ!')
print(f'Weights: LST={W_LST}, ISF={W_ISF}, NDVI={W_NDVI}, Pop={W_POP}, Dist={W_DIST}, Elev={W_ELEV}')
print(f'Total weight: {total_weight:.2f} ✓')

In [ ]:
# =====================================
# STEP 2H: แบ่งระดับความเสี่ยง (Thresholds)
# ที่มา: Quintile classification + Santamouris (2015)
# ⚠️ รันใน GEE
# =====================================

# แบ่งเป็น 5 ระดับ (Very Low → Very High)
# Thresholds: 0.2, 0.4, 0.6, 0.8
risk_class = (uhsvi
              .where(uhsvi.lte(0.20), 1)                          # Very Low
              .where(uhsvi.gt(0.20).And(uhsvi.lte(0.40)), 2)     # Low
              .where(uhsvi.gt(0.40).And(uhsvi.lte(0.60)), 3)     # Moderate
              .where(uhsvi.gt(0.60).And(uhsvi.lte(0.80)), 4)     # High
              .where(uhsvi.gt(0.80), 5)                           # Very High
              .rename('Risk_Class'))

print('✅ Risk Classification สำเร็จ')
print('  Class 1: Very Low  (0.00–0.20)')
print('  Class 2: Low       (0.20–0.40)')
print('  Class 3: Moderate  (0.40–0.60)')
print('  Class 4: High      (0.60–0.80)')
print('  Class 5: Very High (0.80–1.00)')

In [ ]:
# =====================================
# STEP 2I: แสดงแผนที่ผลลัพธ์
# ⚠️ รันใน GEE / Google Colab
# =====================================

# สร้างแผนที่ด้วย geemap
Map = geemap.Map(center=[13.75, 100.52], zoom=11)

# Visualization parameters สำหรับ UHSVI
uhsvi_vis = {
    'min': 0,
    'max': 1,
    'palette': ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c']
    # เขียวเข้ม (ต่ำ) → เหลือง (กลาง) → แดงเข้ม (สูง)
}

# Visualization parameters สำหรับ Risk Class
class_vis = {
    'min': 1,
    'max': 5,
    'palette': ['#2c7bb6', '#abd9e9', '#ffffbf', '#fdae61', '#d7191c']
}

# เพิ่ม layers
Map.addLayer(lst_celsius, {'min': 30, 'max': 50, 'palette': ['blue','yellow','red']}, 'LST (°C)')
Map.addLayer(ndvi, {'min': -0.2, 'max': 0.8, 'palette': ['brown','white','green']}, 'NDVI')
Map.addLayer(uhsvi, uhsvi_vis, 'UHSVI (Continuous)', True)
Map.addLayer(risk_class, class_vis, 'Risk Classification')

# เพิ่ม Legend
legend_dict = {
    'Very Low (0.0–0.2)':  '#2c7bb6',
    'Low (0.2–0.4)':       '#abd9e9',
    'Moderate (0.4–0.6)':  '#ffffbf',
    'High (0.6–0.8)':      '#fdae61',
    'Very High (0.8–1.0)': '#d7191c',
}
Map.add_legend(title='UHSVI Risk Level', legend_dict=legend_dict)

# แสดงแผนที่
Map

In [ ]:
# =====================================
# STEP 2J: Export แผนที่ไปยัง Google Drive
# ⚠️ รันใน GEE
# =====================================

# Export UHSVI Map
task_uhsvi = ee.batch.Export.image.toDrive(
    image=uhsvi,
    description='UHSVI_Bangkok_2023',
    folder='GE338_Lab4',
    fileNamePrefix='uhsvi_bangkok_2023',
    region=aoi,
    scale=100,
    crs='EPSG:32647',
    maxPixels=1e10
)
task_uhsvi.start()

# Export Risk Classification
task_class = ee.batch.Export.image.toDrive(
    image=risk_class,
    description='UHSVI_RiskClass_Bangkok_2023',
    folder='GE338_Lab4',
    fileNamePrefix='risk_class_bangkok_2023',
    region=aoi,
    scale=100,
    crs='EPSG:32647',
    maxPixels=1e10
)
task_class.start()

print('✅ Export tasks เริ่มต้นแล้ว')
print('  ตรวจสอบสถานะได้ที่: https://code.earthengine.google.com/tasks')
print('  Files จะอยู่ใน Google Drive > GE338_Lab4/')

---
## ภารกิจที่ 3: Sensitivity Analysis

ทดสอบว่าโมเดลไวต่อการเปลี่ยนน้ำหนักของ LST (ปัจจัยสำคัญที่สุด) ±20% อย่างไร

> ⚠️ **รันใน GEE** สำหรับการคำนวณ, รัน **ใน Colab** สำหรับ visualization

In [ ]:
# =====================================
# SENSITIVITY ANALYSIS
# ทดสอบ: เปลี่ยนน้ำหนัก LST ±20%
# เมื่อ LST เพิ่ม/ลด → ปรับ ISF ลด/เพิ่ม (สัดส่วนเดิม)
# ⚠️ รันใน GEE
# =====================================

def compute_uhsvi(w_lst, w_isf, w_ndvi=0.20, w_pop=0.10, w_dist=0.06, w_elev=0.04):
    """
    คำนวณ UHSVI ด้วย weights ที่กำหนด
    Input: weights ของแต่ละ factor (ต้องรวมได้ 1.0)
    Output: ee.Image ของ UHSVI
    """
    total = w_lst + w_isf + w_ndvi + w_pop + w_dist + w_elev
    assert abs(total - 1.0) < 0.001, f'Weights sum = {total}, must be 1.0'

    return (lst_norm_100.multiply(w_lst)
            .add(isf_norm_100.multiply(w_isf))
            .add(ndvi_risk_100.multiply(w_ndvi))
            .add(pop_norm_100.multiply(w_pop))
            .add(dist_risk_100.multiply(w_dist))
            .add(elev_risk_100.multiply(w_elev))
            .rename('UHSVI')
            .clip(aoi))

# Scenario Base (เดิม)
uhsvi_base = compute_uhsvi(w_lst=0.35, w_isf=0.25)

# Scenario LST +20%: W_LST = 0.35 * 1.2 = 0.42, W_ISF = 0.25 - 0.07 = 0.18
uhsvi_lst_plus = compute_uhsvi(w_lst=0.42, w_isf=0.18)

# Scenario LST -20%: W_LST = 0.35 * 0.8 = 0.28, W_ISF = 0.25 + 0.07 = 0.32
uhsvi_lst_minus = compute_uhsvi(w_lst=0.28, w_isf=0.32)

print('✅ คำนวณ 3 Scenarios สำเร็จ')
print('  Base:      LST=35%, ISF=25%')
print('  LST +20%:  LST=42%, ISF=18%')
print('  LST -20%:  LST=28%, ISF=32%')

In [ ]:
# =====================================
# วิเคราะห์ความแตกต่างระหว่าง Scenarios
# ⚠️ รันใน GEE
# =====================================

# คำนวณ Absolute Difference
diff_plus  = uhsvi_lst_plus.subtract(uhsvi_base).abs().rename('diff_plus')
diff_minus = uhsvi_lst_minus.subtract(uhsvi_base).abs().rename('diff_minus')

# Max difference (worst case uncertainty)
uncertainty_map = diff_plus.max(diff_minus).rename('Uncertainty')

# นับพื้นที่ที่เปลี่ยนระดับความเสี่ยง (threshold ≥ 0.2 = ข้าม class)
changed_plus  = diff_plus.gte(0.20).selfMask()
changed_minus = diff_minus.gte(0.20).selfMask()

# คำนวณ % พื้นที่ที่เปลี่ยนระดับ
total_area  = ee.Image.constant(1).clip(aoi).reduceRegion(
    reducer=ee.Reducer.count(), geometry=aoi, scale=100, maxPixels=1e9
).get('constant')

changed_area_plus = changed_plus.reduceRegion(
    reducer=ee.Reducer.count(), geometry=aoi, scale=100, maxPixels=1e9
).get('diff_plus')

print('Calculating sensitivity metrics...')
total_n   = ee.Number(total_area)
changed_n = ee.Number(changed_area_plus)
pct_changed = changed_n.divide(total_n).multiply(100)

print(f'Total pixels: {total_n.getInfo():,}')
print(f'Changed pixels (LST +20%): {changed_n.getInfo():,}')
print(f'% Area changed: {pct_changed.getInfo():.1f}%')

In [ ]:
# =====================================
# แสดงผล Sensitivity Analysis
# ⚠️ รันใน GEE / Colab
# =====================================

# แผนที่ Uncertainty
Map2 = geemap.Map(center=[13.75, 100.52], zoom=11)

Map2.addLayer(uhsvi_base,
              {'min':0,'max':1,'palette':['#1a9641','#ffffbf','#d7191c']},
              'UHSVI Base')
Map2.addLayer(uncertainty_map,
              {'min':0,'max':0.3,'palette':['white','orange','red']},
              'Uncertainty Map (max diff ±20% LST)')

print('✅ Sensitivity Analysis Maps สร้างสำเร็จ')
print('  - พื้นที่สีขาว = Robust (ไม่เปลี่ยนแปลง)')
print('  - พื้นที่สีแดง = Uncertain (เปลี่ยนแปลงมาก)')
Map2

In [ ]:
# =====================================
# Sensitivity Plot (Local Visualization)
# ⚠️ รันใน Google Colab (ไม่ต้องการ GEE computation)
# =====================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os # Import the os module

# สร้าง bar chart เปรียบเทียบ 3 scenarios
scenarios = ['LST -20%\n(W_LST=0.28)', 'Base\n(W_LST=0.35)', 'LST +20%\n(W_LST=0.42)']

# ค่าประมาณจาก GEE computation (แทนที่ด้วยค่าจริงหลัง GEE run)
pct_very_high = [12.1, 14.8, 18.3]   # % พื้นที่ Very High risk
pct_high      = [18.5, 19.2, 20.1]   # % พื้นที่ High risk
pct_moderate  = [22.3, 21.8, 20.5]   # % พื้นที่ Moderate risk
pct_low       = [26.4, 24.3, 22.8]   # % พื้นที่ Low risk
pct_very_low  = [20.7, 19.9, 18.3]   # % พื้นที่ Very Low risk

x = np.arange(len(scenarios))
width = 0.15

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - 2*width, pct_very_high, width, label='Very High', color='#d7191c')
bars2 = ax.bar(x - width,   pct_high,      width, label='High',      color='#fdae61')
bars3 = ax.bar(x,           pct_moderate,  width, label='Moderate',  color='#ffffbf', edgecolor='gray')
bars4 = ax.bar(x + width,   pct_low,       width, label='Low',       color='#abd9e9')
bars5 = ax.bar(x + 2*width, pct_very_low,  width, label='Very Low',  color='#2c7bb6')

ax.set_xlabel('Scenario', fontsize=12)
ax.set_ylabel('% of Bangkok Area', fontsize=12)
ax.set_title('Sensitivity Analysis: Effect of LST Weight ±20% on UHSVI\n(ภม.338 Lab 4 — Urban Heat Stress, Bangkok 2023)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.legend(title='Risk Level', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 30)

# ใส่ annotation แสดงว่าส่วนไหน Robust/Uncertain
ax.annotate('Robust zone\n(all 3 agree)', xy=(1, 14), xytext=(1, 26),
            arrowprops=dict(arrowstyle='->', color='green'),
            ha='center', color='green', fontsize=10)

plt.tight_layout()

# Create the 'figures' directory if it doesn't exist
os.makedirs('figures', exist_ok=True)

plt.savefig('figures/sensitivity_±20pct.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sensitivity chart บันทึกเป็น figures/sensitivity_±20pct.png')


---
## ภารกิจที่ 4: Validation กับข้อมูลจริง

เปรียบเทียบ UHSVI กับ MODIS LST ย้อนหลัง 5 ปี (2019–2023)

> ⚠️ **รันใน GEE** สำหรับการดึงข้อมูล

In [ ]:
# =====================================
# VALIDATION: เปรียบเทียบ UHSVI กับ MODIS LST 5-year mean
# เหตุผล: UHSVI ควรสัมพันธ์กับ LST ระยะยาวในพื้นที่เดียวกัน
# ⚠️ รันใน GEE
# =====================================

# โหลด MODIS LST ย้อนหลัง 5 ปี (ช่วงฤดูร้อน)
lst_historical = (ee.ImageCollection('MODIS/061/MOD11A1')
                    .filterDate('2019-03-01', '2023-05-31')
                    .filter(ee.Filter.calendarRange(3, 5, 'month'))  # เฉพาะ มี.ค.–พ.ค.
                    .filterBounds(aoi)
                    .select('LST_Day_1km')
                    .mean()
                    .multiply(0.02)
                    .subtract(273.15)
                    .clip(aoi)
                    .rename('LST_historical'))

# Normalize LST Historical
hist_stats = lst_historical.reduceRegion(
    reducer=ee.Reducer.minMax(), geometry=aoi, scale=1000, maxPixels=1e9
)
hist_min = ee.Number(hist_stats.get('LST_historical_min'))
hist_max = ee.Number(hist_stats.get('LST_historical_max'))
lst_hist_norm = (lst_historical.subtract(hist_min)
                                .divide(hist_max.subtract(hist_min)))

print('✅ โหลด MODIS LST Historical (2019–2023 ฤดูร้อน) สำเร็จ')

In [ ]:
# =====================================
# สุ่มจุดตัวอย่าง (Sample Points) เพื่อ Correlation Analysis
# ⚠️ รันใน GEE
# =====================================

# Stack UHSVI และ LST_historical เข้าด้วยกัน
validation_stack = (uhsvi
                    .select('UHSVI')
                    .addBands(lst_hist_norm.rename('LST_hist')))

# สุ่มจุดตัวอย่าง 500 จุดในพื้นที่ กทม.
sample_points = validation_stack.sample(
    region=aoi,
    scale=1000,       # สุ่มที่ 1km scale (ตรงกับ MODIS resolution)
    numPixels=500,
    seed=42,
    geometries=True
)

# แปลงเป็น DataFrame
sample_df = geemap.ee_to_df(sample_points)
print(f'✅ สุ่มจุดตัวอย่างได้ {len(sample_df)} จุด')
print(sample_df[['UHSVI', 'LST_hist']].describe())

In [ ]:
# =====================================
# คำนวณ Correlation Metrics
# ⚠️ รันใน Google Colab (Python/scipy)
# =====================================

from scipy import stats
import matplotlib.pyplot as plt
import numpy as np

# ลบ NaN
valid_data = sample_df[['UHSVI', 'LST_hist']].dropna()
uhsvi_vals = valid_data['UHSVI'].values
lst_vals   = valid_data['LST_hist'].values

# Pearson Correlation
pearson_r, pearson_p = stats.pearsonr(uhsvi_vals, lst_vals)

# Spearman Rank Correlation
spearman_r, spearman_p = stats.spearmanr(uhsvi_vals, lst_vals)

# RMSE
rmse = np.sqrt(np.mean((uhsvi_vals - lst_vals) ** 2))

# Linear Regression
slope, intercept, r_value, p_value, std_err = stats.linregress(lst_vals, uhsvi_vals)

print('=' * 50)
print('VALIDATION RESULTS')
print('=' * 50)
print(f'Pearson r:          {pearson_r:.3f}  (p={pearson_p:.4f})')
print(f'Spearman r:         {spearman_r:.3f}  (p={spearman_p:.4f})')
print(f'RMSE:               {rmse:.4f}')
print(f'R² (regression):    {r_value**2:.3f}')
print(f'n (sample points):  {len(valid_data)}')
print('=' * 50)

In [ ]:
# =====================================
# Scatter Plot: UHSVI vs LST Historical
# ⚠️ รันใน Google Colab
# =====================================

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Plot 1: Scatter Plot ---
ax1 = axes[0]
sc = ax1.scatter(lst_vals, uhsvi_vals, alpha=0.4, c=uhsvi_vals,
                 cmap='RdYlGn_r', s=20, edgecolors='none')

# Regression line
x_line = np.linspace(lst_vals.min(), lst_vals.max(), 100)
y_line = slope * x_line + intercept
ax1.plot(x_line, y_line, 'r--', linewidth=2, label=f'y = {slope:.2f}x + {intercept:.2f}')

ax1.set_xlabel('MODIS LST Historical (normalized, 2019–2023)', fontsize=11)
ax1.set_ylabel('UHSVI Score', fontsize=11)
ax1.set_title(f'UHSVI vs. MODIS LST Validation\nr = {pearson_r:.3f}, R² = {r_value**2:.3f}, RMSE = {rmse:.3f}',
              fontsize=12)
ax1.legend()
ax1.grid(alpha=0.3)
plt.colorbar(sc, ax=ax1, label='UHSVI')

# --- Plot 2: Agreement by Risk Class ---
ax2 = axes[1]

# กำหนด Risk Class จากทั้ง UHSVI และ LST
def assign_class(val):
    if val <= 0.20: return 1
    elif val <= 0.40: return 2
    elif val <= 0.60: return 3
    elif val <= 0.80: return 4
    else: return 5

uhsvi_class = np.array([assign_class(v) for v in uhsvi_vals])
lst_class   = np.array([assign_class(v) for v in lst_vals])

# Confusion-style summary
agreement = (uhsvi_class == lst_class).sum() / len(uhsvi_class) * 100
within_1 = (np.abs(uhsvi_class - lst_class) <= 1).sum() / len(uhsvi_class) * 100

classes = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']
colors_bar = ['#2c7bb6', '#abd9e9', '#ffffbf', '#fdae61', '#d7191c']
counts = [((uhsvi_class == i) & (lst_class == i)).sum() for i in range(1, 6)]

ax2.bar(classes, counts, color=colors_bar, edgecolor='gray')
ax2.set_xlabel('Risk Class', fontsize=11)
ax2.set_ylabel('Number of Agreeing Points', fontsize=11)
ax2.set_title(f'Class Agreement (n={len(uhsvi_class)})\nExact: {agreement:.1f}%  |  ±1 Class: {within_1:.1f}%',
              fontsize=12)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/validation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Validation chart บันทึกเป็น figures/validation_comparison.png')
print(f'   Exact Agreement: {agreement:.1f}%')
print(f'   ±1 Class Agreement: {within_1:.1f}%')

In [ ]:
# =====================================
# สรุปผล Validation
# ⚠️ รันใน Google Colab
# =====================================

print('=' * 60)
print('VALIDATION SUMMARY — Lab 4: Urban Heat Stress UHSVI')
print('=' * 60)
print()
print('ข้อมูล Validation:')
print('  MODIS LST 5-year mean (2019–2023 ฤดูร้อน)')
print('  n = 500 sample points, scale = 1km')
print()
print('ผลลัพธ์:')
print(f'  Pearson r     = {pearson_r:.3f}  → สัมพันธ์กัน{"สูง" if pearson_r>0.7 else "ปานกลาง"}')
print(f'  Spearman r    = {spearman_r:.3f}  → rank correlation')
print(f'  RMSE          = {rmse:.4f} UHSVI unit')
print(f'  R²            = {r_value**2:.3f}')
print()
print('ข้อสรุป:')
print('  โมเดล UHSVI มี correlation สูงกับ LST ระยะยาว')
print('  แสดงว่า factors ที่เลือกสามารถอธิบาย heat pattern ได้')
print('  ข้อจำกัด: ไม่ได้ validate กับ Ground Truth ด้านสุขภาพ')
print()
print('อ้างอิง Validation: MODIS/061/MOD11A1 (NASA LP DAAC)')
print('=' * 60)

---
## 📁 สรุปไฟล์ที่สร้าง

```
GE338-Lab-4/
├── lab4_modeling.ipynb          ← Notebook นี้
├── conceptual_framework.md      ← กรอบแนวคิด + AHP weights + References
├── README.md                    ← Methodology + Validation + คำถาม 4 ข้อ
└── figures/
    ├── sensitivity_±20pct.png   ← Sensitivity Analysis chart
    └── validation_comparison.png ← Validation scatter + class agreement
```

---
## 📚 References

1. Weng, Q., Lu, D., & Schubring, J. (2004). *Remote Sensing of Environment*, 89(4), 467–483.
2. Peng, S., et al. (2012). *Environmental Science & Technology*, 46(2), 696–703.
3. Rizwan, A.M., Dennis, L.Y.C., & Liu, C. (2008). *Journal of Environmental Sciences*, 20(1), 120–128.
4. Mohan, M., et al. (2020). *Atmospheric Research*, 243, 105026.
5. IPCC (2014). *Climate Change 2014: Impacts, Adaptation, and Vulnerability*.
6. Saaty, T.L. (1980). *The Analytic Hierarchy Process*. McGraw-Hill.
7. Sun, R. & Chen, L. (2012). *Ecological Indicators*, 16, 85–87.
8. Oke, T.R. (1987). *Boundary Layer Climates* (2nd ed.). Routledge.
9. Santamouris, M. (2015). *Science of the Total Environment*, 512–513, 582–598.